### Hyperparameter Tuning and Training for Explainable Boosting Machines (EBM)

This notebook performs hyperparameter tuning for EBMs using interpretML package.


In [1]:
import numpy as np
import pandas as pd
import json
import time
import sys
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, mean_squared_error, mean_absolute_error
from interpret.glassbox import ExplainableBoostingClassifier, ExplainableBoostingRegressor

# Add src to path for imports
project_root = Path.cwd().parent.parent
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import neural_additive_models.data_utils as data_utils
import warnings
warnings.filterwarnings('ignore')


## Configuration


In [2]:
# Dataset configuration
dataset_name = 'OpenML_37_classification'  # Change this to your dataset
is_regression = False  # Set to True for regression

# Hyperparameter search space
hp_search_space = {
    'learning_rate': [0.001, 0.1],
    'max_bins': [16, 32, 64, 128, 256, 512],
    'max_interaction_bins': [8, 16, 32, 64, 128],
    'interactions': [0, 2, 5, 10, 15, 20],
    'outer_bags': [1, 2, 4, 8, 16],
    'inner_bags': [0, 2, 4, 8, 16],
    'min_samples_leaf': [1, 2, 3, 5, 10],
    'max_leaves': [2, 3, 5, 10, 15, 20]
}

# Fixed hyperparameters
fixed_hp = {
    'random_state': 42,
    'n_jobs': -1,
    'early_stopping_rounds': 50,
    'validation_size': 0.2
}

# Tuning parameters
n_trials = 50
random_seed = 42

# Set results directory
project_root = Path.cwd().parent.parent
results_dir = project_root / 'results' / 'hyperparameter_tuning' / 'ebm'
results_dir.mkdir(parents=True, exist_ok=True)


## Load Dataset


In [7]:
# Load dataset
print(f"Loading dataset: {dataset_name}")
data_x, data_y, column_names = data_utils.load_dataset(dataset_name)

# Determine if regression from dataset name
if '_regression' in dataset_name:
    is_regression = True
elif '_classification' in dataset_name:
    is_regression = False

print(f"Dataset shape: {data_x.shape}")
print(f"Target shape: {data_y.shape}")
print(f"Number of features: {data_x.shape[1]}")
print(f"Task type: {'Regression' if is_regression else 'Classification'}")

# Split into train/val/test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    data_x, data_y, test_size=0.2, random_state=random_seed,
    stratify=data_y if not is_regression else None
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.2, random_state=random_seed,
    stratify=y_train_val if not is_regression else None
)

print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")


Loading dataset: OpenML_37_classification
Dataset shape: (768, 8)
Target shape: (768,)
Number of features: 8
Task type: Classification
Train: 491, Val: 123, Test: 154


## Hyperparameter Sampling


In [8]:
def sample_hyperparameters(search_space, random_seed=None):
    """Sample hyperparameters from search space."""
    np.random.seed(random_seed)
    hp = {}
    
    for key, values in search_space.items():
        if isinstance(values, list) and len(values) == 2 and all(isinstance(x, (int, float)) for x in values):
            # Continuous range
            hp[key] = float(np.random.uniform(values[0], values[1]))
        else:
            # Discrete choices
            value = np.random.choice(values)
            hp[key] = value.item() if isinstance(value, np.generic) else value
    
    return hp

# Generate hyperparameter configurations
hyperparameters = []
for trial in range(n_trials):
    trial_seed = random_seed + trial
    hp_config = sample_hyperparameters(hp_search_space, trial_seed)
    hyperparameters.append({
        'trial': trial + 1,
        'hyperparameters': hp_config
    })

print(f"Generated {n_trials} hyperparameter configurations")


Generated 50 hyperparameter configurations


## Hyperparameter Tuning


In [9]:
def train_and_evaluate_ebm(X_train, y_train, X_val, y_val, hyperparameters, is_regression=False):
    """Train EBM and return validation score."""
    
    if is_regression:
        model = ExplainableBoostingRegressor(
            **hyperparameters,
            **fixed_hp
        )
    else:
        model = ExplainableBoostingClassifier(
            **hyperparameters,
            **fixed_hp
        )
    
    # Train
    start_time = time.time()
    model.fit(X_train, y_train)
    training_time = time.time() - start_time
    
    # Predict and evaluate
    y_pred = model.predict(X_val)
    
    if is_regression:
        # RMSE for regression
        score = np.sqrt(mean_squared_error(y_val, y_pred))
    else:
        # AUC for classification
        y_pred_proba = model.predict_proba(X_val)[:, 1]
        score = roc_auc_score(y_val, y_pred_proba)
    
    return score, training_time, model

# Run hyperparameter tuning
print("="*70)
print(f"HYPERPARAMETER TUNING - {n_trials} trials")
print("="*70)

trial_results = []

for trial_data in hyperparameters:
    trial_num = trial_data['trial']
    hp = trial_data['hyperparameters']
    
    print(f"\nTrial {trial_num}/{n_trials}...", end=' ', flush=True)
    
    try:
        score, train_time, model = train_and_evaluate_ebm(
            X_train, y_train, X_val, y_val, hp, is_regression
        )
        
        metric_name = 'RMSE' if is_regression else 'AUC'
        print(f"{metric_name}: {score:.4f} ({train_time:.1f}s)")
        
        trial_results.append({
            'trial': trial_num,
            'hyperparameters': hp,
            'validation_score': score,
            'training_time': train_time,
            'success': True
        })
    except Exception as e:
        print(f"Failed: {str(e)[:50]}")
        trial_results.append({
            'trial': trial_num,
            'hyperparameters': hp,
            'validation_score': None,
            'training_time': None,
            'success': False,
            'error': str(e)
        })

print("\n" + "="*70)
print("TUNING COMPLETE")
print("="*70)


HYPERPARAMETER TUNING - 50 trials

Trial 1/50... AUC: 0.8616 (12.4s)

Trial 2/50... AUC: 0.8442 (0.2s)

Trial 3/50... AUC: 0.8744 (9.5s)

Trial 4/50... AUC: 0.8721 (0.4s)

Trial 5/50... AUC: 0.8331 (13.4s)

Trial 6/50... AUC: 0.8477 (0.3s)

Trial 7/50... AUC: 0.8369 (1.9s)

Trial 8/50... AUC: 0.8669 (127.9s)

Trial 9/50... AUC: 0.8453 (1.1s)

Trial 10/50... AUC: 0.8276 (0.5s)

Trial 11/50... AUC: 0.8660 (0.2s)

Trial 12/50... AUC: 0.8648 (5.2s)

Trial 13/50... AUC: 0.8756 (0.1s)

Trial 14/50... AUC: 0.8718 (0.5s)

Trial 15/50... AUC: 0.8375 (89.0s)

Trial 16/50... AUC: 0.8520 (0.2s)

Trial 17/50... AUC: 0.8206 (559.8s)

Trial 18/50... AUC: 0.8686 (6.2s)

Trial 19/50... AUC: 0.8747 (2.2s)

Trial 20/50... AUC: 0.8183 (4.8s)

Trial 21/50... AUC: 0.8259 (1.2s)

Trial 22/50... AUC: 0.8756 (0.1s)

Trial 23/50... AUC: 0.8279 (2.1s)

Trial 24/50... AUC: 0.8744 (0.3s)

Trial 25/50... AUC: 0.8206 (7.3s)

Trial 26/50... AUC: 0.8384 (0.7s)

Trial 27/50... AUC: 0.8456 (6.1s)

Trial 28/50... AUC: 0.

In [10]:
df_results = pd.DataFrame(trial_results)

# Filter successful trials
df_success = df_results[df_results['success']].copy()

if len(df_success) > 0:
    # Find best hyperparameters
    if is_regression:
        # Lower is better for RMSE
        best_idx = df_success['validation_score'].idxmin()
    else:
        # Higher is better for AUC
        best_idx = df_success['validation_score'].idxmax()
    
    best_trial = df_success.loc[best_idx]
    
    print(f"Best trial: {int(best_trial['trial'])}")
    metric_name = 'RMSE' if is_regression else 'AUC'
    print(f"Best {metric_name}: {best_trial['validation_score']:.4f}")
    print(f"Training time: {best_trial['training_time']:.1f}s")
    print("\nBest hyperparameters:")
    for key, value in best_trial['hyperparameters'].items():
        print(f"  {key}: {value}")
    
    # Save best hyperparameters
    best_hp_file = results_dir / f"best_hp_{dataset_name.replace('/', '_').replace(':', '_')}.json"
    with open(best_hp_file, 'w') as f:
        json.dump(best_trial['hyperparameters'], f, indent=2)
    print(f"\nSaved best hyperparameters to: {best_hp_file}")
    
    # Save all results
    results_file = results_dir / f"tuning_results_{dataset_name.replace('/', '_').replace(':', '_')}.json"
    with open(results_file, 'w') as f:
        json.dump(trial_results, f, indent=2)
    print(f"Saved all results to: {results_file}")
    
    # Display summary statistics
    print("\n" + "="*70)
    print("SUMMARY STATISTICS")
    print("="*70)
    print(f"Successful trials: {len(df_success)}/{n_trials}")
    print(f"Mean {metric_name}: {df_success['validation_score'].mean():.4f}")
    print(f"Std {metric_name}: {df_success['validation_score'].std():.4f}")
    print(f"Mean training time: {df_success['training_time'].mean():.1f}s")
else:
    print("No successful trials!")


Best trial: 45
Best AUC: 0.8840
Training time: 5.1s

Best hyperparameters:
  learning_rate: 0.02105808840926016
  max_bins: 256
  max_interaction_bins: 128
  interactions: 10
  outer_bags: 1
  inner_bags: 2
  min_samples_leaf: 5
  max_leaves: 3

Saved best hyperparameters to: ebm_tuning_results\best_hp_OpenML_37_classification.json
Saved all results to: ebm_tuning_results\tuning_results_OpenML_37_classification.json

SUMMARY STATISTICS
Successful trials: 50/50
Mean AUC: 0.8534
Std AUC: 0.0204
Mean training time: 19.9s


In [ ]:
## Visualize Multi-Seed Training Results


## Train Multiple Models with Different Seeds

Train multiple models with different random seeds to assess stability and get robust performance estimates, similar to NAM training methodology.


In [13]:
if len(df_success) > 0:
    
    best_hp = best_trial['hyperparameters']
    num_seeds = 20
    seed_start = 1
    
    # Combine train and val for final training
    X_train_final = np.vstack([X_train, X_val])
    y_train_final = np.concatenate([y_train, y_val])
    
    print("="*70)
    print(f"TRAINING {num_seeds} MODELS WITH DIFFERENT SEEDS")
    print("="*70)
    
    seed_results = []
    
    for seed_idx in range(num_seeds):
        seed = seed_start + seed_idx
        
        print(f"\nTraining model {seed_idx + 1}/{num_seeds} (seed={seed})...", end=' ', flush=True)
        
        try:
            # Create model with seed-specific random_state
            if is_regression:
                model = ExplainableBoostingRegressor(
                    **best_hp,
                    random_state=seed,
                    n_jobs=fixed_hp['n_jobs'],
                    early_stopping_rounds=fixed_hp['early_stopping_rounds'],
                    validation_size=fixed_hp['validation_size']
                )
            else:
                model = ExplainableBoostingClassifier(
                    **best_hp,
                    random_state=seed,
                    n_jobs=fixed_hp['n_jobs'],
                    early_stopping_rounds=fixed_hp['early_stopping_rounds'],
                    validation_size=fixed_hp['validation_size']
                )
            
            # Train
            start_time = time.time()
            model.fit(X_train_final, y_train_final)
            training_time = time.time() - start_time
            
            # Evaluate on test set
            y_test_pred = model.predict(X_test)
            
            if is_regression:
                test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
                test_mae = mean_absolute_error(y_test, y_test_pred)
                metric_name = 'RMSE'
                metric_value = test_rmse
                print(f"{metric_name}: {test_rmse:.4f}, MAE: {test_mae:.4f} ({training_time:.1f}s)")
            else:
                y_test_proba = model.predict_proba(X_test)[:, 1]
                test_auc = roc_auc_score(y_test, y_test_proba)
                metric_name = 'AUC'
                metric_value = test_auc
                print(f"{metric_name}: {test_auc:.4f} ({training_time:.1f}s)")
            
            seed_results.append({
                'seed': seed,
                'test_score': metric_value,
                'training_time': training_time,
                'success': True
            })
            
        except Exception as e:
            print(f"Failed: {str(e)[:50]}")
            seed_results.append({
                'seed': seed,
                'test_score': None,
                'training_time': None,
                'success': False,
                'error': str(e)
            })
    
    # Summary statistics
    df_seed_results = pd.DataFrame(seed_results)
    df_seed_success = df_seed_results[df_seed_results['success']].copy()
    
    if len(df_seed_success) > 0:
        mean_score = df_seed_success['test_score'].mean()
        std_score = df_seed_success['test_score'].std()
        
        print("\n" + "="*70)
        print("MULTI-SEED TRAINING SUMMARY")
        print("="*70)
        print(f"Successful runs: {len(df_seed_success)}/{num_seeds}")
        print(f"Test {metric_name}: {mean_score:.4f} ± {std_score:.4f}")
        print(f"Min test {metric_name}: {df_seed_success['test_score'].min():.4f}")
        print(f"Max test {metric_name}: {df_seed_success['test_score'].max():.4f}")
        print(f"Mean training time: {df_seed_success['training_time'].mean():.1f}s")
        
        # Save multi-seed results
        multi_seed_file = results_dir / f"multi_seed_results_{dataset_name.replace('/', '_').replace(':', '_')}.json"
        with open(multi_seed_file, 'w') as f:
            json.dump(seed_results, f, indent=2)
        print(f"\nSaved multi-seed results to: {multi_seed_file}")
    else:
        print("\nNo successful multi-seed runs!")
else:
    print("Cannot train multi-seed models - no successful hyperparameter tuning trials")


TRAINING 20 MODELS WITH DIFFERENT SEEDS

Training model 1/20 (seed=1)... AUC: 0.8143 (3.7s)

Training model 2/20 (seed=2)... AUC: 0.7887 (3.2s)

Training model 3/20 (seed=3)... AUC: 0.8170 (3.8s)

Training model 4/20 (seed=4)... AUC: 0.8276 (3.7s)

Training model 5/20 (seed=5)... AUC: 0.8115 (3.5s)

Training model 6/20 (seed=6)... AUC: 0.8083 (3.0s)

Training model 7/20 (seed=7)... AUC: 0.8257 (2.2s)

Training model 8/20 (seed=8)... AUC: 0.8083 (2.1s)

Training model 9/20 (seed=9)... AUC: 0.8309 (3.7s)

Training model 10/20 (seed=10)... AUC: 0.8111 (3.5s)

Training model 11/20 (seed=11)... AUC: 0.8256 (5.0s)

Training model 12/20 (seed=12)... AUC: 0.8170 (3.3s)

Training model 13/20 (seed=13)... AUC: 0.8087 (2.9s)

Training model 14/20 (seed=14)... AUC: 0.8344 (3.8s)

Training model 15/20 (seed=15)... AUC: 0.8124 (2.6s)

Training model 16/20 (seed=16)... AUC: 0.8376 (3.0s)

Training model 17/20 (seed=17)... AUC: 0.8298 (2.4s)

Training model 18/20 (seed=18)... AUC: 0.8126 (3.4s)

Train